# APEX Optimizer Tutorial: Math Problem Solving

This tutorial demonstrates how to use **APEX** (Analysis-based Prompt Engineering eXpert) to improve GPT-5 Mini's performance on AIME math problems through systematic prompt optimization.

APEX analyzes failures, recognizes success patterns, generates hypotheses, and validates improvements empirically.

## Configuration

All modifiable parameters in one place for easy adjustment:

In [1]:
# API Configuration
api_key = 'sk-12345'# Will prompt if not set
base_url = 'https://nexus-master.lmndstaging.com'

# Student Model Configuration (model being optimized)
student_model = "litellm_proxy/openai/gpt-5-mini"
student_base_url = base_url  # Optional custom API endpoint
student_temperature = 0.0  # Deterministic for math
student_reasoning_effort = 'minimal'

# Analysis Model Configuration (for failure analysis and hypotheses)
analysis_model = "litellm_proxy/openai/gpt-5"
analysis_base_url = base_url  # Optional custom API endpoint
analysis_temperature = 1.0  # Creative for hypothesis generation
analysis_reasoning_effort = 'minimal'

# APEX Optimization Settings
max_iterations = 50
num_hypotheses = 3
train_sample_size = 10
success_threshold = 1.0
convergence_patience = 10
num_threads = 50
seed = 42
verbosity = "detailed"

# MLflow Tracking (Optional)
use_mlflow = True
mlflow_tracking_uri = "http://localhost:5005"
mlflow_experiment_name = "APEX-AIME-Math"

## Setup

Import dependencies and configure language models:

In [2]:
import os
import dspy
from dspy.adapters import JSONAdapter

if api_key is None:
    api_key = os.getenv("OPENAI_API_KEY") or input("Enter your OpenAI API key: ")

# Configure student model
student_kwargs = {
    "model": student_model,
    "api_key": api_key,
    "temperature": student_temperature,
}
if student_base_url:
    student_kwargs["base_url"] = student_base_url
if student_reasoning_effort:
    student_kwargs["reasoning_effort"] = student_reasoning_effort

student_lm = dspy.LM(**student_kwargs)

# Configure analysis model
analysis_kwargs = {
    "model": analysis_model,
    "api_key": api_key,
    "temperature": analysis_temperature,
}
if analysis_base_url:
    analysis_kwargs["base_url"] = analysis_base_url
if analysis_reasoning_effort:
    analysis_kwargs["reasoning_effort"] = analysis_reasoning_effort

analysis_lm = dspy.LM(**analysis_kwargs)

analysis_adapter = JSONAdapter()
hypothesis_adapter = JSONAdapter()

dspy.configure(lm=student_lm)
dspy.settings.configure(num_threads=num_threads)

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Dataset

Load AIME problems (American Invitational Mathematics Examination):

In [3]:
from datasets import load_dataset
import random

def init_dataset():
    train_split = load_dataset("AI-MO/aimo-validation-aime")['train']
    train_split = [
        dspy.Example({
            "problem": x['problem'],
            "solution": x['solution'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in train_split
    ]
    
    random.Random(0).shuffle(train_split)
    tot_num = len(train_split)

    test_split = load_dataset("MathArena/aime_2025")['train']
    test_split = [
        dspy.Example({
            "problem": x['problem'],
            "answer": x['answer'],
        }).with_inputs("problem")
        for x in test_split
    ]

    train_set = train_split[: int(0.5 * tot_num)]
    val_set = train_split[int(0.5 * tot_num):]
    test_set = test_split * 5

    return train_set, val_set, test_set

train_set, val_set, test_set = init_dataset()

print(f"Training: {len(train_set)} | Validation: {len(val_set)} | Test: {len(test_set)}")

Training: 45 | Validation: 45 | Test: 150


Example problem:

In [4]:
print("Problem:", train_set[0]['problem'])
print("\nAnswer:", train_set[0]['answer'])

Problem: In isosceles trapezoid $ABCD$, parallel bases $\overline{AB}$ and $\overline{CD}$ have lengths $500$ and $650$, respectively, and $AD=BC=333$. The angle bisectors of $\angle{A}$ and $\angle{D}$ meet at $P$, and the angle bisectors of $\angle{B}$ and $\angle{C}$ meet at $Q$. Find $PQ$.

Answer: 242


## Program

Define a Chain of Thought program:

In [5]:
class GenerateResponse(dspy.Signature):
    """Solve the problem and provide the answer in the correct format."""
    problem = dspy.InputField()
    answer = dspy.OutputField()

program = dspy.ChainOfThought(GenerateResponse)

## Metrics

Define evaluation metrics:

In [6]:
def metric(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        return 0
    return int(correct_answer == llm_answer)

In [7]:
def metric_with_feedback(example, prediction, trace=None, pred_name=None, pred_trace=None):
    correct_answer = int(example['answer'])
    written_solution = example.get('solution', '')
    
    try:
        llm_answer = int(prediction.answer)
    except ValueError:
        feedback_text = (
            f"The final answer must be a valid integer. "
            f"You responded with '{prediction.answer}', which couldn't be parsed. "
            f"The correct answer is '{correct_answer}'."
        )
        
        if written_solution:
            feedback_text += f" Here's the full solution:\n{written_solution}"
        
        return dspy.Prediction(score=0, feedback=feedback_text)

    score = int(correct_answer == llm_answer)
    
    if score == 1:
        feedback_text = f"Correct! The answer is '{correct_answer}'."
    else:
        feedback_text = f"Incorrect. The correct answer is '{correct_answer}'."

    if written_solution:
        feedback_text += f" Here's the full solution:\n{written_solution}"

    return dspy.Prediction(score=score, feedback=feedback_text)

## Baseline Evaluation

Evaluate the unoptimized program:

In [8]:
eval_kwargs = dict(
    num_threads=num_threads,
    display_progress=True,
    display_table=5,
    provide_traceback=False,
)

evaluate = dspy.Evaluate(
    devset=test_set,
    metric=metric,
    **eval_kwargs,
)

print("Evaluating baseline...")
baseline_result = evaluate(program)

print(f"\nBaseline Performance: {baseline_result.score / 100.:.1%}")

Evaluating baseline...
Average Metric: 80.00 / 150 (53.3%): 100%|██████████| 150/150 [00:00<00:00, 228.53it/s]

2025/10/17 22:03:57 INFO dspy.evaluate.evaluate: Average Metric: 80 / 150 (53.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We interpret 17_b = b+7 and 97_b = 9b+7. We need b+7 to divide 9b+...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"Set up affine coordinates with A=(0,0), B=(1,0), C=(0,1). Points o...",441,✔️ [0]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,"We must count assignments of 3 labeled flavors (C, V, S) to 9 dist...",16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer solutions (x,y) in [-100,100] satisfying 12x^2 - x...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,Divisibility by 22 means divisible by 2 and 11. Units digit must b...,279,✔️ [1]



Baseline Performance: 53.3%


## APEX Optimization

Optimize the program with APEX:

In [ ]:
from dspy.teleprompt.apex import APEX

optimizer = APEX(
    metric=metric_with_feedback,
    analysis_lm=analysis_lm,
    hypothesis_lm=analysis_lm,
    analysis_adapter=analysis_adapter,
    hypothesis_adapter=hypothesis_adapter,
    max_iterations=max_iterations,
    num_hypotheses=num_hypotheses,
    num_eval_runs=1,
    train_sample=train_sample_size,
    success_threshold=success_threshold,
    convergence_patience=convergence_patience,
    num_threads=num_threads,
    verbosity=verbosity,
    seed=seed,
    use_mlflow=use_mlflow,
    mlflow_tracking_uri=mlflow_tracking_uri,
    mlflow_experiment_name=mlflow_experiment_name,
)

print("Starting optimization...")
optimized_program = optimizer.compile(
    student=program,
    trainset=train_set,
    valset=val_set,
)

print("\nOptimization complete!")

2025/10/17 22:03:57 INFO dspy.teleprompt.apex.apex: APEX: MLflow tracking enabled


Starting optimization...


2025/10/17 22:03:57 INFO dspy.teleprompt.apex.apex: APEX: running with num_threads=50
2025/10/17 22:03:57 INFO dspy.teleprompt.apex.apex: APEX: Configuration - max_iterations=50, num_hypotheses=3, success_threshold=1.00, convergence_patience=10
2025/10/17 22:03:57 INFO dspy.teleprompt.apex.apex: APEX: Using seed=42 for reproducibility
2025/10/17 22:03:57 INFO dspy.teleprompt.apex.apex: APEX: Evaluating initial baseline on validation set


Processed 45 / 45 examples: 100%|██████████| 45/45 [00:02<00:00, 22.29it/s]

2025/10/17 22:03:59 INFO dspy.teleprompt.apex.apex: APEX: Initial baseline score=0.5111


...

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{\n  "re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
2025/10/18 03:48:13 INFO dspy.teleprompt.apex.apex: APEX: iteration 28 hypothesis score=0.6000
2025/10/18 03:48:13 INFO dspy.teleprompt.apex.apex: APEX: hypothesis details → {'observation': 'Failures show missing explicit small-case/base-case validation leading to wrong general formulas in probability and counting; successes benefit from structured-methodology and verification but lack a c

2025/10/18 03:48:14 INFO dspy.teleprompt.apex.apex: APEX: Iteration 28 best score: 0.6222
2025/10/18 03:48:14 INFO dspy.teleprompt.apex.apex: APEX: Score improvements from baseline: [-0.06666666666666665, -0.2, -0.022222222222222254]
2025/10/18 03:48:14 INFO dspy.teleprompt.apex.apex: APEX: No improvement (10/10 patience)
2025/10/18 03:48:14 INFO dspy.teleprompt.apex.apex: APEX: Stopping due to convergence patience reached
2025/10/18 03:48:14 INFO dspy.teleprompt.apex.apex: APEX: ✓ Optimization complete | 28 iterations | Reason: patience
2025/10/18 03:48:14 INFO dspy.teleprompt.apex.apex: APEX: Final score: 0.6222 (+0.1111 from baseline 0.5111)
2025/10/18 03:48:14 INFO dspy.teleprompt.apex.apex: APEX: Summary - evaluated 108 candidates from 81 hypotheses
2025/10/18 03:48:14 INFO dspy.teleprompt.apex.apex: APEX: Best score trajectory across iterations: [0.5777777777777777, 0.5777777777777777, 0.5777777777777777, 0.6, 0.6222222222222222, 0.6222222222222222, 0.6222222222222222, 0.62222222

🏃 View run peaceful-ant-287 at: http://localhost:5005/#/experiments/1/runs/076971fc10e244b0a0aca7be2ca7117b
🧪 View experiment at: http://localhost:5005/#/experiments/1

╔═══════════════════════════════════════════════════════════════╗
║                    APEX Optimization Summary                     ║
╠═══════════════════════════════════════════════════════════════╣
║ Total Iterations: 28                                              ║
║ Hypotheses Tested: 81                                             ║
║ Candidates Evaluated: 108                                         ║
╠═══════════════════════════════════════════════════════════════╣
║ Initial Score: 0.5111                                             ║
║ Final Score: 0.6222                                               ║
║ Improvement: +0.1111                                              ║
╠═══════════════════════════════════════════════════════════════╣
║ Iter │ Train F/S │ Hypotheses │ Best Score │ Δ from prev ║
╟──────┼─────────

[Trace(trace_id=tr-3e3426bf83b22b34f4fa70002f9bef60), Trace(trace_id=tr-862bf74ebb810793b7cd41e4b56cdfa2), Trace(trace_id=tr-5d5f3e94c47de0a4828202438ac44fa6), Trace(trace_id=tr-968f7d8ff8811b1a61aa5c0018ed33cf), Trace(trace_id=tr-14069316d4a1b6b107e9fed2f3c64ce2), Trace(trace_id=tr-44dc2505777e1b2c7193b4b21968d4b0), Trace(trace_id=tr-c6789d6e7f22a6d847a9cd87d92ac348), Trace(trace_id=tr-6078dafb7446ba81da865d688cf6b74b), Trace(trace_id=tr-b2b39ed516b9e01dc23fe45844ef3ab5), Trace(trace_id=tr-0f364ee814d0ee4f6e398cbd0c06e1e0)]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='{\n  "re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpec

Inspect the optimized prompt:

In [10]:
print("Optimized Prompt:")
print("=" * 50)
print(optimized_program.predict.signature.instructions)
print("=" * 50)

Optimized Prompt:
Solve using a clear method and validate on a small case before finalizing. Then output only the final numeric answer.

Base-case verification (mandatory):
- After deriving your formula/recurrence/structure, compute at least one small instance (e.g., n=1,2 or the simplest nontrivial parameters) explicitly and confirm agreement.
- If disagreement occurs, revise the derivation and repeat until agreement.

Method cues:
- Probability/Expectation: DP or staged probability with linearity; base-case n=1,2 agreement required.
- Counting/Diophantine: derive necessary-and-sufficient structure, parameterize/exhaustively count; check the smallest admissible parameter values.
- Geometry: exact tools (radical axis/PoP/similarity), no decimals; recompute givens and confirm symbolic relations.
- Regular-polygon rectangles: enumerate orientation/slope families including non-antipodal; verify coverage by testing smallest polygons where cases are transparent.

Final-answer contract (mand

## Final Evaluation

Evaluate the optimized program:

In [11]:
print("Evaluating optimized program...")
optimized_result = evaluate(optimized_program)

print(f"\n{'='*50}")
print(f"Baseline:  {baseline_result.score/100.:.1%}")
print(f"Optimized: {optimized_result.score/100.:.1%}")
print(f"Improvement: {(optimized_result.score - baseline_result.score)/100.:.1%}")
print(f"{'='*50}")

Evaluating optimized program...
Average Metric: 4.00 / 4 (100.0%):   2%|▏         | 3/150 [00:10<06:46,  2.77s/it]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content='[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpec

Average Metric: 7.00 / 7 (100.0%):   5%|▍         | 7/150 [00:12<02:18,  1.03it/s]

/Users/aviram.kofman/Documents/Projects/dspy/.venv/lib/python3.10/site-packages/pydantic/main.py:463: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 9 fields but got 6: Expected `Message` - serialized value may not be as expected [input_value=Message(content="[[ ## re...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Average Metric: 15.00 / 15 (100.0%):   9%|▉         | 14/150 [00:16<01:26,  1.56it/s]

...

Average Metric: 86.00 / 150 (57.3%): : 151it [02:12,  1.14it/s]                       

2025/10/18 03:50:27 INFO dspy.evaluate.evaluate: Average Metric: 86 / 150 (57.3%)


,problem,example_answer,reasoning,pred_answer,metric
0,Find the sum of all integer bases $b>9$ for which $17_b$ is a divi...,70,We interpret 17_b and 97_b in base b. Their base-10 values: - 17_b...,70,✔️ [1]
1,"On $\triangle ABC$ points $A, D, E$, and $B$ lie in that order on ...",588,"We have triangle ABC with points on AB: A - D - E - B with AD=4, D...","{""reasoning"":""We place A as origin and express positions along AB ...",✔️ [0]
2,The 9 members of a baseball team went to an ice-cream parlor after...,16,We need number of assignments of 9 labeled players to 3 flavors (C...,16,✔️ [1]
3,"Find the number of ordered pairs $(x,y)$, where both $x$ and $y$ a...",117,"We need integer pairs (x,y) with -100 ≤ x,y ≤ 100 satisfying 12x^2...",117,✔️ [1]
4,There are $8!= 40320$ eight-digit positive integers that use each ...,279,We must count 8-digit permutations of digits 1..8 divisible by 22....,279,✔️ [1]



Baseline:  53.3%
Optimized: 57.3%
Improvement: 4.0%


[Trace(trace_id=tr-1a3598c9e422f4de528c7bb661b2dab4), Trace(trace_id=tr-0a8a4fd269b245493951ebac487dc1c7), Trace(trace_id=tr-9beae619f47e79e13639e36c9af5cf44)]

## Optimization Insights

Examine the optimization process:

In [12]:
if hasattr(optimized_program, 'apex_result'):
    result = optimized_program.apex_result
    
    print("Summary:")
    print(f"  Iterations: {len(result.iterations)}")
    print(f"  Candidates evaluated: {len(result.all_candidates)}")
    print(f"  Stop reason: {result.stopped_after}")
    print(f"  Best score: {result.best_candidate.overall_score:.4f}")
    
    print("\nIteration Progress:")
    for it in result.iterations:
        print(f"  Iteration {it.iteration}: {it.num_failures} failures, {len(it.hypotheses)} hypotheses, {len(it.candidates)} candidates")
    
    if result.best_candidate.hypothesis:
        h = result.best_candidate.hypothesis
        print(f"\nBest Hypothesis:")
        print(f"  Strategy: {h.strategy if hasattr(h, 'strategy') else 'N/A'}")
        print(f"  Impact Score: {h.impact_score if hasattr(h, 'impact_score') else 'N/A'}")

Summary:
  Iterations: 28
  Candidates evaluated: 109
  Stop reason: patience
  Best score: 0.6222

Iteration Progress:
  Iteration 1: 2 failures, 3 hypotheses, 4 candidates
  Iteration 2: 2 failures, 3 hypotheses, 4 candidates
  Iteration 3: 6 failures, 3 hypotheses, 4 candidates
  Iteration 4: 2 failures, 3 hypotheses, 4 candidates
  Iteration 5: 7 failures, 3 hypotheses, 4 candidates
  Iteration 6: 5 failures, 3 hypotheses, 4 candidates
  Iteration 7: 3 failures, 3 hypotheses, 4 candidates
  Iteration 8: 2 failures, 3 hypotheses, 4 candidates
  Iteration 9: 3 failures, 3 hypotheses, 4 candidates
  Iteration 10: 3 failures, 3 hypotheses, 4 candidates
  Iteration 11: 2 failures, 3 hypotheses, 4 candidates
  Iteration 12: 2 failures, 3 hypotheses, 4 candidates
  Iteration 13: 7 failures, 3 hypotheses, 4 candidates
  Iteration 14: 2 failures, 3 hypotheses, 4 candidates
  Iteration 15: 5 failures, 3 hypotheses, 4 candidates
  Iteration 16: 3 failures, 3 hypotheses, 4 candidates
  Iterati

## Conclusion

APEX systematically optimizes prompts through:
1. Analyzing failures to understand root causes
2. Recognizing successful patterns
3. Generating data-driven hypotheses
4. Validating improvements empirically

Try adjusting the configuration parameters to explore different optimization strategies.